# Manage temporary files

You need to create temporary files or directories for intermediate data, caching, or testing. You want them to be cleaned up automatically when you are done.

This guide covers the `tempfile` module and common patterns for temporary resource management.

## Creating a temporary file with `tempfile.NamedTemporaryFile`

The `tempfile.NamedTemporaryFile` function creates a temporary file that is automatically deleted when the `with` block ends.

In [ ]:
import tempfile
from pathlib import Path

with tempfile.NamedTemporaryFile(
    mode="w", suffix=".txt", encoding="utf-8"
) as f:
    f.write("Temporary data\n")
    f.flush()
    print(f"Temporary file: {f.name}")
    print(f"File exists during with block: {Path(f.name).exists()}")

print(f"File exists after with block: {Path(f.name).exists()}")

## Keeping a temporary file after closing

Use `delete=False` when you need the file to outlive the `with` block. You are then responsible for deleting it yourself.

In [ ]:
import tempfile
from pathlib import Path

with tempfile.NamedTemporaryFile(
    mode="w", suffix=".txt", delete=False, encoding="utf-8"
) as f:
    f.write("This file persists.\n")
    temp_path = Path(f.name)

print(f"File still exists: {temp_path.exists()}")
print(f"Content: {temp_path.read_text(encoding='utf-8')}")

temp_path.unlink()
print(f"After manual cleanup: {temp_path.exists()}")

## Creating a temporary directory

The `tempfile.TemporaryDirectory` function creates a temporary directory. The entire directory and its contents are automatically deleted when the `with` block ends.

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmpdir:
    tmpdir_path = Path(tmpdir)
    (tmpdir_path / "data.txt").write_text("Hello", encoding="utf-8")
    (tmpdir_path / "config.txt").write_text("Setting=1", encoding="utf-8")

    print(f"Temporary directory: {tmpdir}")
    print(f"Contents: {[p.name for p in sorted(tmpdir_path.iterdir())]}")

print(f"Directory exists after with block: {Path(tmpdir).exists()}")

## Using `tempfile.mkdtemp()` for manual management

When you need more control over the lifetime of a temporary directory, use `tempfile.mkdtemp()`. You must clean it up yourself.

In [ ]:
import shutil
import tempfile
from pathlib import Path

tmpdir = tempfile.mkdtemp()
tmpdir_path = Path(tmpdir)
try:
    (tmpdir_path / "output.txt").write_text("Result", encoding="utf-8")
    print(f"Working in: {tmpdir}")
    content = (tmpdir_path / "output.txt").read_text(encoding="utf-8")
    print(f"Content: {content}")
finally:
    shutil.rmtree(tmpdir)
    print(f"Cleaned up: {not tmpdir_path.exists()}")

## Practical example: safe file writing

A common pattern is to write data to a temporary file first, then rename it to the target path. This prevents data corruption if the write is interrupted.

In [ ]:
import tempfile
from pathlib import Path


def safe_write(filepath: str | Path, content: str) -> None:
    """Write content to a file safely using a temporary file.

    Writes to a temporary file first, then renames it to the target
    path. This prevents data corruption if the write is interrupted.

    Args:
        filepath: The path to the target file.
        content: The text content to write.
    """
    path = Path(filepath)
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(
        mode="w",
        dir=path.parent,
        suffix=".tmp",
        delete=False,
        encoding="utf-8",
    ) as f:
        f.write(content)
        temp_path = Path(f.name)
    temp_path.rename(path)


safe_write("important-data.txt", "Critical information\n")
print(Path("important-data.txt").read_text(encoding="utf-8"))
Path("important-data.txt").unlink()

## Using temporary files in tests

The `tempfile` module is essential for writing tests that work with files. Temporary files ensure tests are self-contained and clean up after themselves.

In [ ]:
import tempfile
import unittest
from pathlib import Path


def count_words(filepath: str | Path) -> int:
    """Count words in a file.

    Args:
        filepath: The path to the file.

    Returns:
        The total number of words.
    """
    path = Path(filepath)
    with path.open("r", encoding="utf-8") as f:
        return sum(len(line.split()) for line in f)


class TestCountWords(unittest.TestCase):
    def test_count_words_in_file(self):
        with tempfile.NamedTemporaryFile(
            mode="w", suffix=".txt", delete=False, encoding="utf-8"
        ) as f:
            f.write("one two three\nfour five\n")
            temp_path = f.name
        try:
            result = count_words(temp_path)
            self.assertEqual(result, 5)
        finally:
            Path(temp_path).unlink()


suite = unittest.TestLoader().loadTestsFromTestCase(TestCountWords)
unittest.TextTestRunner(verbosity=2).run(suite)

## Key takeaways

- `tempfile.NamedTemporaryFile` creates temporary files that are automatically deleted
- Use `delete=False` when the file needs to persist beyond the `with` block
- `tempfile.TemporaryDirectory` creates temporary directories with automatic cleanup
- Write to a temporary file and rename for safe file updates
- Use `tempfile` in tests to avoid leaving test files behind